Main file generation



In [ ]:
import numpy as np
from scipy.io import wavfile

# --------------------------------------------------
# Parameters
# --------------------------------------------------

INPUT_FILE = "input.wav"

FRAME_SIZE = 1024
OVERLAP = 0.50       # 75% overlap

# --------------------------------------------------
# Read WAV file
# --------------------------------------------------

sample_rate, audio = wavfile.read("/sample-speech-1m.wav")

print("Sampling rate:", sample_rate, "Hz")
print("Number of samples:", len(audio))

# --------------------------------------------------
# Convert stereo to mono if necessary
# --------------------------------------------------

if audio.ndim > 1:
    audio = audio.mean(axis=1)

# Convert to floating point
audio = audio.astype(np.float64)

# Normalize if integer WAV
if np.max(np.abs(audio)) > 0:
    audio = audio / np.max(np.abs(audio))

# --------------------------------------------------
# Calculate hop size
# --------------------------------------------------

hop_size = int(FRAME_SIZE * (1 - OVERLAP))

print("Frame size:", FRAME_SIZE)
print("Overlap:", OVERLAP * 100, "%")
print("Hop size:", hop_size)

# --------------------------------------------------
# Create Hann window
# --------------------------------------------------

hann_window = np.hanning(FRAME_SIZE)

# --------------------------------------------------
# Extract overlapping frames
# --------------------------------------------------

frames = []

for start in range(0, len(audio) - FRAME_SIZE + 1, hop_size):

    # Extract N samples
    frame = audio[start:start + FRAME_SIZE]

    # Apply Hann window
    windowed_frame = frame * hann_window

    frames.append(windowed_frame)

# Convert list to NumPy array
frames = np.array(frames)

# --------------------------------------------------
# Display information
# --------------------------------------------------

print("Number of frames:", len(frames))
print("Shape of frames:", frames.shape)

# Example: first frame
print("\nFirst frame:")
print(frames[0])

# --------------------------------------------------
# Convert frames to 16-bit signed Q15
# and write to .mem file
# --------------------------------------------------

MEM_FILE = "/audio_frames.mem"

# Convert floating point [-1, 1] to signed 16-bit
q15_frames = np.clip(frames, -1.0, 1.0)

q15_frames = np.where(
    q15_frames < 0,
    q15_frames * 32768,
    q15_frames * 32767
)

q15_frames = np.round(q15_frames).astype(np.int16)

# Write samples to .mem file
with open(MEM_FILE, "w") as f:
    for frame in q15_frames:
        for sample in frame:

            # Convert signed int16 to 16-bit two's complement
            hex_value = int(sample) & 0xFFFF

            f.write(f"{hex_value:04X}\n")

print("MEM file created:", MEM_FILE)
print("Total samples written:", q15_frames.size)

Sampling rate: 44100 Hz
Number of samples: 2646016
Frame size: 1024
Overlap: 50.0 %
Hop size: 512
Number of frames: 5167
Shape of frames: (5167, 1024)

First frame:
[-0.00000000e+00 -1.16945506e-08 -4.67777613e-08 ... -4.74014648e-06
 -1.17583391e-06 -0.00000000e+00]
MEM file created: /audio_frames.mem
Total samples written: 5291008


**Notes ROM table**

In [ ]:
import math

ROM_FILE = "tuning_rom.mem"

# 88 piano keys:
# A0 = MIDI 21
# C8 = MIDI 108

with open(ROM_FILE, "w") as f:

    for midi_note in range(21, 109):

        # Equal temperament, A4 = MIDI 69 = 440 Hz
        frequency = 440.0 * (2 ** ((midi_note - 69) / 12))

        # Store nearest integer Hz
        frequency_int = round(frequency)

        # 16-bit hexadecimal
        f.write(f"{frequency_int:04X}\n")

print("ROM created:", ROM_FILE)
print("Entries:", 88)

ROM created: tuning_rom.mem
Entries: 88


In [ ]:
!ls -l /content/tuning_rom.mem

-rw-r--r-- 1 root root 440 Aug 17 09:35 /content/tuning_rom.mem


In [ ]:
from google.colab import files

files.download("/content/tuning_rom.mem")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Sine Wave Generation**

In [ ]:
import numpy as np

# ============================================================
# PARAMETERS
# ============================================================

SAMPLE_RATE = 44100
FRAME_SIZE = 1024
NUM_FRAMES = 4
NUM_SAMPLES = FRAME_SIZE * NUM_FRAMES

BIN = 10
AMPLITUDE = 0.5

# Exact bin-aligned frequency
FREQUENCY = BIN * SAMPLE_RATE / FRAME_SIZE

print(f"Frequency = {FREQUENCY:.9f} Hz")
print(f"Samples   = {NUM_SAMPLES}")

# ============================================================
# GENERATE SINE WAVE
# ============================================================

n = np.arange(NUM_SAMPLES)

samples = AMPLITUDE * np.sin(
    2 * np.pi * BIN * n / FRAME_SIZE
)

# Convert to signed 16-bit
samples_int = np.round(
    samples * 32767
).astype(np.int16)

# ============================================================
# WRITE VERILOG .MEM FILE
# ============================================================

with open("audio_frames.mem", "w") as f:

    for sample in samples_int:

        # Convert signed int16 to two's-complement hex
        value = int(sample) & 0xFFFF

        f.write(f"{value:04X}\n")

# ============================================================
# DISPLAY INFORMATION
# ============================================================

print("Generated: audio_frames.mem")
print(f"Min sample = {samples_int.min()}")
print(f"Max sample = {samples_int.max()}")

print("\nFirst 16 samples:")

for i in range(16):
    print(
        f"{i:4d}: "
        f"{samples_int[i]:7d}  "
        f"0x{(int(samples_int[i]) & 0xFFFF):04X}"
    )

Frequency = 430.664062500 Hz
Samples   = 4096
Generated: audio_frames.mem
Min sample = -16384
Max sample = 16384

First 16 samples:
   0:       0  0x0000
   1:    1005  0x03ED
   2:    2006  0x07D6
   3:    2999  0x0BB7
   4:    3981  0x0F8D
   5:    4948  0x1354
   6:    5896  0x1708
   7:    6823  0x1AA7
   8:    7723  0x1E2B
   9:    8595  0x2193
  10:    9434  0x24DA
  11:   10237  0x27FD
  12:   11002  0x2AFA
  13:   11726  0x2DCE
  14:   12406  0x3076
  15:   13039  0x32EF


In [ ]:
import numpy as np

# ============================================================
# Parameters
# ============================================================

N = 1024
AMPLITUDE = 12000

# Harmonic bins
k1 = 10
k2 = 20
k3 = 30
k5 = 50

# Relative amplitudes
A1 = 1.0
A2 = 0.50
A3 = 0.30
A5 = 0.15


# ============================================================
# Generate waveform
# ============================================================

n = np.arange(N)

x = (
    A1 * np.sin(2 * np.pi * k1 * n / N)
    + A2 * np.sin(2 * np.pi * k2 * n / N)
    + A3 * np.sin(2 * np.pi * k3 * n / N)
    + A5 * np.sin(2 * np.pi * k5 * n / N)
)


# ============================================================
# Normalize so we don't overflow 16-bit signed range
# ============================================================

x = x / np.max(np.abs(x))

x = np.round(x * AMPLITUDE).astype(np.int16)


# ============================================================
# Write .mem file
# ============================================================

with open("harmonic_test.mem", "w") as f:

    for sample in x:

        # Convert signed int16 to 16-bit two's complement
        value = int(sample) & 0xFFFF

        f.write(f"{value:04X}\n")


print("Generated harmonic_test.mem")
print("Number of samples:", len(x))
print("Maximum:", np.max(x))
print("Minimum:", np.min(x))

**Composite Sine Waveform**

In [ ]:
import numpy as np

# ============================================================
# Configuration
# ============================================================

N = 1024                  # Number of samples / FFT size

INPUT_MEM = "harmonic.mem"
FFT_MEM = "fft_expected.mem"

# Four FFT-bin frequencies
bins = [40, 100, 180, 250]

# Amplitudes of the four sine waves
amplitudes = [4000, 3000, 2500, 2000]

# Phase shifts in degrees
# Non-zero phases produce both real and imaginary FFT components
phases_deg = [30, 45, 60, 90]

# ============================================================
# Generate the input signal
# ============================================================

n = np.arange(N)

signal = np.zeros(N, dtype=float)

for k, amplitude, phase_deg in zip(
        bins, amplitudes, phases_deg):

    phase = np.deg2rad(phase_deg)

    signal += amplitude * np.sin(
        2 * np.pi * k * n / N + phase
    )

# ============================================================
# Convert to signed 16-bit
# ============================================================

signal = np.clip(signal, -32768, 32767)

signal = np.round(signal).astype(np.int16)

# ============================================================
# Write input.mem
#
# Each line = one 16-bit signed sample
# ============================================================

with open(INPUT_MEM, "w") as f:

    for sample in signal:

        # Convert signed integer to 16-bit two's complement
        value = int(sample) & 0xFFFF

        f.write(f"{value:04X}\n")

print(f"Created: {INPUT_MEM}")

# ============================================================
# Calculate FFT
# ============================================================

X = np.fft.fft(signal)

# ============================================================
# Scale FFT output to signed 16-bit
#
# Both real and imaginary components are scaled using
# the same factor.
# ============================================================

max_value = max(
    np.max(np.abs(X.real)),
    np.max(np.abs(X.imag))
)

if max_value == 0:
    scale = 1.0
else:
    scale = 32767.0 / max_value

X_scaled = X * scale

# ============================================================
# Convert real and imaginary parts to integers
# ============================================================

real = np.round(X_scaled.real).astype(np.int32)
imag = np.round(X_scaled.imag).astype(np.int32)

# Saturate to signed 16-bit
real = np.clip(real, -32768, 32767)
imag = np.clip(imag, -32768, 32767)

# ============================================================
# Pack FFT output
#
# 31                     16 15                      0
# ┌────────────────────────┬────────────────────────┐
# │      IMAGINARY         │         REAL           │
# │       [31:16]          │        [15:0]          │
# └────────────────────────┴────────────────────────┘
#
# ============================================================

with open(FFT_MEM, "w") as f:

    for r, i in zip(real, imag):

        # Convert signed 16-bit to two's complement
        r16 = int(r) & 0xFFFF
        i16 = int(i) & 0xFFFF

        # Imaginary in upper 16 bits
        # Real in lower 16 bits
        packed = (i16 << 16) | r16

        f.write(f"{packed:08X}\n")

print(f"Created: {FFT_MEM}")

# ============================================================
# Display information
# ============================================================

print("\n========================================")
print("FFT Configuration")
print("========================================")

print(f"FFT size        : {N}")
print(f"Frequency bins  : {bins}")
print(f"Amplitudes      : {amplitudes}")
print(f"Phases (deg)    : {phases_deg}")
print(f"FFT scale       : {scale}")

# ============================================================
# Display significant FFT bins
# ============================================================

print("\n========================================")
print("Significant FFT bins")
print("========================================")

threshold = 100

for k in range(N):

    if abs(real[k]) > threshold or abs(imag[k]) > threshold:

        print(
            f"Bin {k:4d}: "
            f"Real = {real[k]:7d}, "
            f"Imag = {imag[k]:7d}"
        )

# ============================================================
# Download files
#
# This section works in Google Colab.
# ============================================================

try:

    from google.colab import files

    print("\n========================================")
    print("Downloading files...")
    print("========================================")

    files.download(INPUT_MEM)
    files.download(FFT_MEM)

except ImportError:

    print("\nNot running in Google Colab.")
    print("Files are available in the current directory:")
    print(f"  {INPUT_MEM}")
    print(f"  {FFT_MEM}")

Created: harmonic.mem
Created: fft_expected.mem

FFT Configuration
FFT size        : 1024
Frequency bins  : [40, 100, 180, 250]
Amplitudes      : [4000, 3000, 2500, 2000]
Phases (deg)    : [30, 45, 60, 90]
FFT scale       : 0.01847463223574858

Significant FFT bins
Bin   40: Real =   18918, Imag =  -32767
Bin  100: Real =   20066, Imag =  -20065
Bin  180: Real =   20480, Imag =  -11824
Bin  250: Real =   18918, Imag =       0
Bin  774: Real =   18918, Imag =       0
Bin  844: Real =   20480, Imag =   11824
Bin  924: Real =   20066, Imag =   20065
Bin  984: Real =   18918, Imag =   32767



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**FFT Testing**

In [ ]:
import numpy as np

INPUT_MEM = "harmonic.mem"
OUTPUT_MEM = "cordic_output.mem"


# ============================================================
# SETTINGS
# ============================================================

# Empirically determined from Vivado vs Python comparison
# Vivado magnitude ≈ Python magnitude × 0.05286
MAG_SCALE = 0.05286

# Vivado CORDIC phase format:
# phase_code = phase_radians × 8192
PHASE_SCALE = 8192.0


# ============================================================
# Read 16-bit signed values from .mem
# ============================================================

def read_mem(filename):
    samples = []

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            # Ignore blank lines and comments
            if not line or line.startswith("//"):
                continue

            value = int(line, 16)

            # Convert 16-bit two's complement to signed integer
            if value & 0x8000:
                value -= 0x10000

            samples.append(value)

    return np.array(samples, dtype=np.int16)


# ============================================================
# Signed integer -> unsigned 16-bit representation
# ============================================================

def to_uint16(value):
    return int(value) & 0xFFFF


# ============================================================
# Read input
# ============================================================

x = read_mem(INPUT_MEM)

N = len(x)

print(f"Number of samples: {N}")


# ============================================================
# FFT
# ============================================================

X = np.fft.fft(x)


# ============================================================
# Normalize FFT real/imaginary values
#
# This is your original Python normalization.
# The magnitude scaling is applied separately below so that
# the FFT normalization and CORDIC magnitude scaling remain
# conceptually separate.
# ============================================================

max_value = max(
    np.max(np.abs(X.real)),
    np.max(np.abs(X.imag))
)

if max_value == 0:
    fft_scale = 1.0
else:
    fft_scale = 32767.0 / max_value

X_scaled = X * fft_scale


# ============================================================
# Convert FFT real/imaginary values to signed 16-bit
# ============================================================

real = np.round(X_scaled.real).astype(np.int32)
imag = np.round(X_scaled.imag).astype(np.int32)

real = np.clip(real, -32768, 32767)
imag = np.clip(imag, -32768, 32767)


# ============================================================
# CORDIC: Cartesian -> Polar
#
# magnitude = sqrt(Re² + Im²)
# phase     = atan2(Im, Re)
# ============================================================

magnitude = np.sqrt(
    real.astype(np.float64)**2 +
    imag.astype(np.float64)**2
)

phase = np.arctan2(
    imag.astype(np.float64),
    real.astype(np.float64)
)


# ============================================================
# VIVADO MAGNITUDE SCALING
#
# Based on the comparison:
#
#       Vivado magnitude
#       ---------------- ≈ 0.05286
#       Python magnitude
#
# Examples:
#
# 1501 / 28377 ≈ 0.05289
# 1250 / 23648 ≈ 0.05286
# 1000 / 18918 ≈ 0.05286
#
# ============================================================

magnitude_vivado = magnitude * MAG_SCALE


# ============================================================
# Convert magnitude to 16-bit unsigned value
# ============================================================

magnitude_16 = np.round(magnitude_vivado).astype(np.int32)

magnitude_16 = np.clip(
    magnitude_16,
    0,
    32767
)


# ============================================================
# VIVADO PHASE FORMAT
#
# Vivado output appears to use:
#
#       phase_code = phase_radians × 8192
#
# Therefore:
#
#       0 rad       -> 0
#       +pi/2       -> +12868
#       -pi/2       -> -12868
#       +pi         -> +25736
#       -pi         -> -25736
#
# ============================================================

phase_16 = np.round(
    phase * PHASE_SCALE
).astype(np.int32)

phase_16 = np.clip(
    phase_16,
    -32768,
    32767
)


# ============================================================
# Pack CORDIC output
#
# [31:16] = phase
# [15:0]  = magnitude
# ============================================================

packed = []

for mag, ph in zip(magnitude_16, phase_16):

    mag16 = to_uint16(mag)
    ph16 = to_uint16(ph)

    value = (ph16 << 16) | mag16

    packed.append(value)


# ============================================================
# Write output .mem
# ============================================================

with open(OUTPUT_MEM, "w") as f:

    for value in packed:
        f.write(f"{value:08X}\n")


# ============================================================
# Information
# ============================================================

print("==============================================")
print("CORDIC OUTPUT GENERATED")
print("==============================================")
print(f"Input file       : {INPUT_MEM}")
print(f"Output file      : {OUTPUT_MEM}")
print(f"FFT size         : {N}")
print(f"FFT scale        : {fft_scale}")
print(f"Magnitude scale  : {MAG_SCALE}")
print(f"Phase scale      : {PHASE_SCALE}")
print("Output format    : [31:16] Phase | [15:0] Magnitude")
print("==============================================")


# ============================================================
# Verify selected bins
# ============================================================

print("\nSelected bins:")

for k in [40, 101, 181, 251]:

    if k < N:

        print(
            f"BIN {k:3d}: "
            f"Re={real[k]:7d}  "
            f"Im={imag[k]:7d}  "
            f"Mag={magnitude_16[k]:5d}  "
            f"Phase={phase_16[k]:6d}  "
            f"HEX={packed[k]:08X}"
        )


# ============================================================
# Download .mem file in Google Colab
# ============================================================

from google.colab import files

files.download(OUTPUT_MEM)

Number of samples: 1024
CORDIC OUTPUT GENERATED
Input file       : harmonic.mem
Output file      : cordic_output.mem
FFT size         : 1024
FFT scale        : 0.01847463223574858
Magnitude scale  : 0.05286
Phase scale      : 8192.0
Output format    : [31:16] Phase | [15:0] Magnitude

Selected bins:
BIN  40: Re=  18918  Im= -32767  Mag= 2000  Phase= -8579  HEX=DE7D07D0
BIN 101: Re=      0  Im=      0  Mag=    0  Phase=     0  HEX=00000000
BIN 181: Re=      0  Im=      0  Mag=    0  Phase=     0  HEX=00000000
BIN 251: Re=      0  Im=      0  Mag=    0  Phase=     0  HEX=00000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**CORDIC TESTING**

In [ ]:
import numpy as np

# ============================================================
# FILES
# ============================================================

CORDIC_INPUT_MEM = "cordic_output.mem"
ROM_FILE = "tuning_rom.mem"
OUTPUT_MEM = "spectral_transform_output.mem"


# ============================================================
# SYSTEM PARAMETERS
# ============================================================

FS = 44100


# ============================================================
# CONVERSION FUNCTIONS
# ============================================================

def signed16(value):
    """
    Convert 16-bit two's complement value to signed integer.
    """
    value = int(value) & 0xFFFF

    if value & 0x8000:
        value -= 0x10000

    return value


def uint16(value):
    """
    Convert integer to unsigned 16-bit representation.
    """
    return int(value) & 0xFFFF


# ============================================================
# READ CORDIC #1 OUTPUT
#
# Format:
#
# [31:16] = Phase (signed 16-bit)
# [15:0]  = Magnitude (unsigned 16-bit)
# ============================================================

def read_cordic_output(filename):

    magnitude = []
    phase = []

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            # Ignore blank lines and comments
            if not line or line.startswith("//"):
                continue

            # Read complete 32-bit hexadecimal value
            value = int(line, 16)

            # ------------------------------------------------
            # Extract magnitude
            # ------------------------------------------------

            mag = value & 0xFFFF

            # ------------------------------------------------
            # Extract phase
            # ------------------------------------------------

            phase_bits = (value >> 16) & 0xFFFF

            ph = signed16(phase_bits)

            magnitude.append(mag)
            phase.append(ph)

    return (
        np.array(magnitude, dtype=np.int32),
        np.array(phase, dtype=np.int32)
    )


# ============================================================
# READ FREQUENCY ROM
#
# ROM entries are hexadecimal.
#
# Example:
#
# 001C -> 28 Hz
# 01B8 -> 440 Hz
# 06E0 -> 1760 Hz
#
# ============================================================

def read_frequency_rom(filename):

    frequencies = []

    with open(filename, "r") as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            # Ignore blank lines and comments
            if not line or line.startswith("//"):
                continue

            try:

                # ROM values are hexadecimal
                value = int(line, 16)

                frequencies.append(value)

            except ValueError:

                print(
                    f"WARNING: Invalid ROM entry at line "
                    f"{line_number}: {line}"
                )

    return np.array(
        frequencies,
        dtype=np.float64
    )


# ============================================================
# READ CORDIC #1 OUTPUT
# ============================================================

magnitude, phase = read_cordic_output(
    CORDIC_INPUT_MEM
)

N = len(magnitude)

if N == 0:
    raise ValueError(
        "CORDIC input file contains no valid data."
    )


print("==============================================")
print("CORDIC #1 OUTPUT LOADED")
print("==============================================")

print(f"Number of FFT bins = {N}")


# ============================================================
# READ 88-NOTE FREQUENCY ROM
# ============================================================

rom_frequencies = read_frequency_rom(
    ROM_FILE
)

if len(rom_frequencies) == 0:
    raise ValueError(
        "Frequency ROM contains no valid entries."
    )

print(f"ROM entries = {len(rom_frequencies)}")

if len(rom_frequencies) != 88:
    print(
        f"WARNING: Expected 88 ROM entries, "
        f"but found {len(rom_frequencies)}"
    )


# ============================================================
# SPECTRAL ANALYSIS
#
# For a real-valued time-domain signal:
#
# Bins 1 to N/2-1 contain positive frequencies.
#
# Bin 0 = DC
# Bin N/2 = Nyquist
# ============================================================

HALF_N = N // 2

analysis_magnitude = magnitude[:HALF_N].copy()

# Ignore DC
analysis_magnitude[0] = 0


# ============================================================
# FIND DOMINANT PEAK BIN
# ============================================================

peak_bin = int(
    np.argmax(analysis_magnitude)
)

peak_magnitude = int(
    magnitude[peak_bin]
)


# ============================================================
# CALCULATE DETECTED FREQUENCY
#
# f = bin × Fs / N
# ============================================================

bin_resolution = FS / N

detected_frequency = (
    peak_bin * bin_resolution
)


# ============================================================
# COMPARE DETECTED FREQUENCY WITH ALL ROM ENTRIES
#
# Find:
#
# min |ROM_frequency - detected_frequency|
# ============================================================

frequency_difference = np.abs(
    rom_frequencies - detected_frequency
)

closest_rom_index = int(
    np.argmin(frequency_difference)
)

target_frequency = float(
    rom_frequencies[closest_rom_index]
)

closest_difference = float(
    frequency_difference[closest_rom_index]
)


# ============================================================
# CALCULATE TRANSFORMATION RATIO
#
# ratio =
#
#     target_frequency
#     ----------------
#     detected_frequency
# ============================================================

if detected_frequency == 0:

    pitch_ratio = 1.0

else:

    pitch_ratio = (
        target_frequency /
        detected_frequency
    )


# ============================================================
# DISPLAY SPECTRAL ANALYSIS RESULTS
# ============================================================

print("\n==============================================")
print("SPECTRAL ANALYSIS RESULTS")
print("==============================================")

print(f"FFT Size                = {N}")
print(f"Sampling Frequency      = {FS} Hz")
print(
    f"Bin Resolution          = "
    f"{bin_resolution:.6f} Hz"
)

print(f"\nPeak Bin                = {peak_bin}")
print(f"Peak Magnitude          = {peak_magnitude}")

print(
    f"Detected Frequency      = "
    f"{detected_frequency:.6f} Hz"
)

print(
    f"\nClosest ROM Index       = "
    f"{closest_rom_index}"
)

print(
    f"Closest ROM Frequency   = "
    f"{target_frequency:.6f} Hz"
)

print(
    f"Frequency Difference    = "
    f"{closest_difference:.6f} Hz"
)

print(
    f"\nTRANSFORMATION RATIO    = "
    f"{pitch_ratio:.8f}"
)


# ============================================================
# SPECTRAL TRANSFORM
#
# Input:
#
# old_bin
#
# Output:
#
# new_bin = round(old_bin × transformation_ratio)
#
# ============================================================

new_magnitude = np.zeros(
    N,
    dtype=np.int32
)

new_phase = np.zeros(
    N,
    dtype=np.int32
)


# ============================================================
# PRESERVE DC
# ============================================================

new_magnitude[0] = magnitude[0]

new_phase[0] = phase[0]


# ============================================================
# TRANSFORM POSITIVE FREQUENCY BINS
#
# Only bins:
#
# 1 to N/2 - 1
# ============================================================

for old_bin in range(1, HALF_N):

    # Calculate destination bin
    new_bin = int(
        round(
            old_bin * pitch_ratio
        )
    )

    # --------------------------------------------------------
    # Only use valid positive-frequency bins
    # --------------------------------------------------------

    if 0 < new_bin < HALF_N:

        # ----------------------------------------------------
        # Multiple source bins may map to the same destination.
        #
        # Keep the strongest spectral component.
        # ----------------------------------------------------

        if magnitude[old_bin] > new_magnitude[new_bin]:

            new_magnitude[new_bin] = (
                magnitude[old_bin]
            )

            new_phase[new_bin] = (
                phase[old_bin]
            )


# ============================================================
# HANDLE NYQUIST BIN
# ============================================================

if N % 2 == 0:

    new_magnitude[HALF_N] = (
        magnitude[HALF_N]
    )

    new_phase[HALF_N] = (
        phase[HALF_N]
    )


# ============================================================
# RECONSTRUCT NEGATIVE FREQUENCY HALF
#
# For a real input signal:
#
# X[N-k] = conjugate(X[k])
#
# Therefore:
#
# Magnitude[N-k] = Magnitude[k]
#
# Phase[N-k] = -Phase[k]
# ============================================================

for k in range(1, HALF_N):

    new_magnitude[N - k] = (
        new_magnitude[k]
    )

    new_phase[N - k] = (
        -new_phase[k]
    )


# ============================================================
# SATURATE OUTPUT
# ============================================================

new_magnitude = np.clip(
    new_magnitude,
    0,
    32767
)

new_phase = np.clip(
    new_phase,
    -32768,
    32767
)


# ============================================================
# PACK OUTPUT
#
# [31:16] = Phase
# [15:0]  = Magnitude
# ============================================================

packed = []

for mag, ph in zip(
    new_magnitude,
    new_phase
):

    mag16 = uint16(mag)

    ph16 = uint16(ph)

    value = (
        (ph16 << 16)
        |
        mag16
    )

    packed.append(value)


# ============================================================
# WRITE OUTPUT MEMORY FILE
# ============================================================

with open(OUTPUT_MEM, "w") as f:

    for value in packed:

        f.write(
            f"{value:08X}\n"
        )


# ============================================================
# CALCULATE EXPECTED TRANSFORMED PEAK
# ============================================================

transformed_peak_bin = int(
    round(
        peak_bin * pitch_ratio
    )
)

transformed_frequency = (
    transformed_peak_bin *
    bin_resolution
)


# ============================================================
# DISPLAY TRANSFORM RESULTS
# ============================================================

print("\n==============================================")
print("SPECTRAL TRANSFORM COMPLETE")
print("==============================================")

print(
    f"Original Peak Bin       = "
    f"{peak_bin}"
)

print(
    f"Original Frequency      = "
    f"{detected_frequency:.6f} Hz"
)

print(
    f"Target Frequency        = "
    f"{target_frequency:.6f} Hz"
)

print(
    f"Transformation Ratio    = "
    f"{pitch_ratio:.8f}"
)

print(
    f"New Peak Bin            = "
    f"{transformed_peak_bin}"
)

print(
    f"Approx New Frequency    = "
    f"{transformed_frequency:.6f} Hz"
)

print(
    f"\nOutput written to: {OUTPUT_MEM}"
)


# ============================================================
# DOWNLOAD OUTPUT FILE
# ============================================================

from google.colab import files

files.download(OUTPUT_MEM)

CORDIC #1 OUTPUT LOADED
Number of FFT bins = 1024
ROM entries = 88

SPECTRAL ANALYSIS RESULTS
FFT Size                = 1024
Sampling Frequency      = 44100 Hz
Bin Resolution          = 43.066406 Hz

Peak Bin                = 40
Peak Magnitude          = 2000
Detected Frequency      = 1722.656250 Hz

Closest ROM Index       = 72
Closest ROM Frequency   = 1760.000000 Hz
Frequency Difference    = 37.343750 Hz

TRANSFORMATION RATIO    = 1.02167800

SPECTRAL TRANSFORM COMPLETE
Original Peak Bin       = 40
Original Frequency      = 1722.656250 Hz
Target Frequency        = 1760.000000 Hz
Transformation Ratio    = 1.02167800
New Peak Bin            = 41
Approx New Frequency    = 1765.722656 Hz

Output written to: spectral_transform_output.mem


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**CORDIC#2 TESTING**

In [ ]:
import numpy as np

# ============================================================
# FILES
# ============================================================

INPUT_MEM = "spectral_transform_output.mem"
OUTPUT_MEM = "cordic2_output.mem"


# ============================================================
# PHASE FORMAT
#
# Input spectral_transform_output.mem:
#
# [31:16] = signed phase
# [15:0]  = unsigned magnitude
#
# Based on your CORDIC #1 phase representation:
#
# phase_code = (theta / pi) * 32768
#
# Therefore:
#
# theta = phase_code * pi / 32768
# ============================================================

PHASE_TO_RAD = np.pi / 32768.0


# ============================================================
# 16-BIT HELPERS
# ============================================================

def signed16(x):
    x = int(x) & 0xFFFF
    if x & 0x8000:
        x -= 0x10000
    return x


def uint16(x):
    return int(x) & 0xFFFF


# ============================================================
# READ INPUT
#
# IMPORTANT:
# Each line corresponds directly to one FFT bin.
# Bin index is simply the line number.
# No sorting, shifting, or remapping is performed.
# ============================================================

magnitudes = []
phase_codes = []

with open(INPUT_MEM, "r") as f:

    for line in f:

        line = line.strip()

        if not line or line.startswith("//"):
            continue

        value = int(line, 16)

        magnitude = value & 0xFFFF

        phase_bits = (value >> 16) & 0xFFFF
        phase = signed16(phase_bits)

        magnitudes.append(magnitude)
        phase_codes.append(phase)


magnitude = np.array(magnitudes, dtype=np.int64)
phase_code = np.array(phase_codes, dtype=np.int64)

N = len(magnitude)

if N == 0:
    raise ValueError("No valid data found.")

print("==============================================")
print("INPUT LOADED")
print("==============================================")
print(f"Number of bins = {N}")


# ============================================================
# INPUT DOMINANT BIN
#
# This MUST remain the same after CORDIC #2 in terms of
# bin position, unless the magnitude changes due to saturation.
# ============================================================

input_peak_bin = int(np.argmax(magnitude))

print(f"Input dominant bin = {input_peak_bin}")
print(f"Input magnitude    = {magnitude[input_peak_bin]}")


# ============================================================
# SPECTRUM_TO_CORDIC
#
# No bin manipulation.
#
# Only convert phase representation:
#
# theta = phase_code * pi / 32768
# ============================================================

theta = (
    phase_code.astype(np.float64)
    * PHASE_TO_RAD
)


# ============================================================
# CORDIC #2
#
# Polar -> Cartesian
#
# real = magnitude * cos(theta)
# imag = magnitude * sin(theta)
#
# Use ROUNDING here, not truncation.
# ============================================================

real_float = (
    magnitude.astype(np.float64)
    * np.cos(theta)
)

imag_float = (
    magnitude.astype(np.float64)
    * np.sin(theta)
)

real = np.round(real_float).astype(np.int64)
imag = np.round(imag_float).astype(np.int64)


# ============================================================
# SATURATE TO SIGNED 16-BIT
# ============================================================

real = np.clip(
    real,
    -32768,
    32767
)

imag = np.clip(
    imag,
    -32768,
    32767
)


# ============================================================
# PACK OUTPUT
#
# [31:16] = Imaginary
# [15:0]  = Real
#
# IMPORTANT:
# packed[k] always corresponds to input bin k.
# ============================================================

packed = []

for k in range(N):

    r = int(real[k])
    i = int(imag[k])

    real_bits = uint16(r)
    imag_bits = uint16(i)

    packed_value = (
        (imag_bits << 16)
        | real_bits
    )

    packed.append(packed_value)


# ============================================================
# WRITE OUTPUT
# ============================================================

with open(OUTPUT_MEM, "w") as f:

    for value in packed:
        f.write(f"{value:08X}\n")


# ============================================================
# OUTPUT MAGNITUDE CHECK
#
# Calculate magnitude from generated Cartesian output.
# This lets us verify which output bin is dominant.
# ============================================================

output_magnitude = np.sqrt(
    real.astype(np.float64) ** 2
    +
    imag.astype(np.float64) ** 2
)

output_peak_bin = int(np.argmax(output_magnitude))


# ============================================================
# RESULTS
# ============================================================

print()
print("==============================================")
print("SPECTRUM_TO_CORDIC + CORDIC #2 COMPLETE")
print("==============================================")

print(f"Input dominant bin  = {input_peak_bin}")
print(f"Output dominant bin = {output_peak_bin}")

print()

if input_peak_bin == output_peak_bin:
    print("PASS: Dominant bin position preserved.")
else:
    print("WARNING: Dominant bin changed.")


# ============================================================
# DEBUG BINS AROUND THE PEAK
# ============================================================

print()
print("PEAK REGION:")
print()

start = max(0, min(input_peak_bin, output_peak_bin) - 3)
end = min(N, max(input_peak_bin, output_peak_bin) + 4)

for k in range(start, end):

    print(
        f"BIN {k:4d} | "
        f"IN_MAG={magnitude[k]:6d} | "
        f"PHASE={phase_code[k]:7d} | "
        f"REAL={real[k]:7d} | "
        f"IMAG={imag[k]:7d} | "
        f"OUT_MAG={output_magnitude[k]:9.2f} | "
        f"HEX={packed[k]:08X}"
    )


# ============================================================
# DOWNLOAD OUTPUT
# ============================================================

from google.colab import files

files.download(OUTPUT_MEM)

INPUT LOADED
Number of bins = 1024
Input dominant bin = 41
Input magnitude    = 2000

SPECTRUM_TO_CORDIC + CORDIC #2 COMPLETE
Input dominant bin  = 41
Output dominant bin = 41

PASS: Dominant bin position preserved.

PEAK REGION:

BIN   38 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000
BIN   39 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000
BIN   40 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000
BIN   41 | IN_MAG=  2000 | PHASE=  -8579 | REAL=   1361 | IMAG=  -1466 | OUT_MAG=  2000.37 | HEX=FA460551
BIN   42 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000
BIN   43 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000
BIN   44 | IN_MAG=     0 | PHASE=      0 | REAL=      0 | IMAG=      0 | OUT_MAG=     0.00 | HEX=00000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**IFFT Testing**

In [ ]:
import numpy as np

# ============================================================
# FILES
# ============================================================

INPUT_MEM = "cordic2_output.mem"
OUTPUT_MEM = "ifft_output.mem"


# ============================================================
# SETTINGS
# ============================================================

FFT_SIZE = 1024


# ============================================================
# 16-BIT CONVERSION FUNCTIONS
# ============================================================

def signed16(value):
    """
    Convert unsigned 16-bit value to signed integer.
    """

    value = int(value) & 0xFFFF

    if value & 0x8000:
        value -= 0x10000

    return value


def uint16(value):
    """
    Convert signed integer to unsigned 16-bit
    two's complement representation.
    """

    return int(value) & 0xFFFF


# ============================================================
# READ CORDIC #2 OUTPUT
#
# Input format:
#
# [31:16] = Imaginary
# [15:0]  = Real
# ============================================================

def read_complex_mem(filename):

    real_values = []
    imag_values = []

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            # Ignore blank lines and comments
            if not line:
                continue

            if line.startswith("//"):
                continue

            value = int(line, 16)

            # ------------------------------------------------
            # Real part
            #
            # [15:0]
            # ------------------------------------------------

            real_bits = value & 0xFFFF

            real = signed16(real_bits)


            # ------------------------------------------------
            # Imaginary part
            #
            # [31:16]
            # ------------------------------------------------

            imag_bits = (value >> 16) & 0xFFFF

            imag = signed16(imag_bits)


            real_values.append(real)
            imag_values.append(imag)


    return (
        np.array(real_values, dtype=np.int64),
        np.array(imag_values, dtype=np.int64)
    )


# ============================================================
# LOAD INPUT
# ============================================================

real_in, imag_in = read_complex_mem(INPUT_MEM)

N = len(real_in)

print("==============================================")
print("CORDIC #2 OUTPUT LOADED")
print("==============================================")

print(f"Number of samples = {N}")


# ============================================================
# CHECK FFT SIZE
# ============================================================

if N != FFT_SIZE:

    raise ValueError(
        f"Expected {FFT_SIZE} samples, "
        f"but found {N}"
    )


# ============================================================
# CREATE COMPLEX INPUT
# ============================================================

X = (
    real_in.astype(np.float64)
    +
    1j * imag_in.astype(np.float64)
)


# ============================================================
# PERFORM IFFT
#
# NumPy ifft automatically includes:
#
#          1
# y[n] = ----- Σ X[k] exp(j2πkn/N)
#          N
#
# This corresponds to the mathematically normalized IFFT.
# ============================================================

x_ifft = np.fft.ifft(X)


# ============================================================
# EXTRACT REAL AND IMAGINARY COMPONENTS
# ============================================================

real_float = x_ifft.real

imag_float = x_ifft.imag


# ============================================================
# QUANTIZATION
#
# Your Vivado FFT/IFFT configuration uses:
#
# Rounding Mode = Truncation
#
# Therefore truncate toward zero.
# ============================================================

real_out = np.trunc(
    real_float
).astype(np.int64)


imag_out = np.trunc(
    imag_float
).astype(np.int64)


# ============================================================
# SATURATE TO SIGNED 16-BIT
# ============================================================

real_out = np.clip(
    real_out,
    -32768,
    32767
)


imag_out = np.clip(
    imag_out,
    -32768,
    32767
)


# ============================================================
# PACK OUTPUT
#
# Output format:
#
# [31:16] = Imaginary
# [15:0]  = Real
# ============================================================

packed = []

for r, i in zip(real_out, imag_out):

    real_bits = uint16(r)

    imag_bits = uint16(i)

    value = (
        (imag_bits << 16)
        |
        real_bits
    )

    packed.append(value)


# ============================================================
# WRITE IFFT OUTPUT
# ============================================================

with open(OUTPUT_MEM, "w") as f:

    for value in packed:

        f.write(f"{value:08X}\n")


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("==============================================")
print("IFFT COMPLETE")
print("==============================================")

print(f"Input file  : {INPUT_MEM}")
print(f"Output file : {OUTPUT_MEM}")
print(f"IFFT size   : {N}")


# ============================================================
# DISPLAY FIRST 10 OUTPUTS
# ============================================================

print()
print("FIRST 10 IFFT OUTPUTS:")
print()

for k in range(min(10, N)):

    print(
        f"SAMPLE {k:4d} | "
        f"REAL={real_out[k]:7d} | "
        f"IMAG={imag_out[k]:7d} | "
        f"HEX={packed[k]:08X}"
    )


# ============================================================
# CHECK IMAGINARY RESIDUAL
#
# For a properly conjugate-symmetric spectrum,
# the time-domain output should be almost entirely real.
# ============================================================

max_imaginary = np.max(
    np.abs(imag_out)
)

print()
print("==============================================")
print("IFFT VERIFICATION")
print("==============================================")

print(
    f"Maximum imaginary residual = "
    f"{max_imaginary}"
)

if max_imaginary <= 2:

    print(
        "PASS: Output is effectively real."
    )

else:

    print(
        "WARNING: Significant imaginary component exists."
    )

print("==============================================")


# ============================================================
# DOWNLOAD OUTPUT
# ============================================================

from google.colab import files

files.download(OUTPUT_MEM)

CORDIC #2 OUTPUT LOADED
Number of samples = 1024

IFFT COMPLETE
Input file  : cordic2_output.mem
Output file : ifft_output.mem
IFFT size   : 1024

FIRST 10 IFFT OUTPUTS:

SAMPLE    0 | REAL=      9 | IMAG=      0 | HEX=00000009
SAMPLE    1 | REAL=      8 | IMAG=      0 | HEX=00000008
SAMPLE    2 | REAL=      3 | IMAG=      0 | HEX=00000003
SAMPLE    3 | REAL=      2 | IMAG=      0 | HEX=00000002
SAMPLE    4 | REAL=      3 | IMAG=      0 | HEX=00000003
SAMPLE    5 | REAL=      2 | IMAG=      0 | HEX=00000002
SAMPLE    6 | REAL=      0 | IMAG=      0 | HEX=00000000
SAMPLE    7 | REAL=      0 | IMAG=      0 | HEX=00000000
SAMPLE    8 | REAL=      0 | IMAG=      0 | HEX=00000000
SAMPLE    9 | REAL=      0 | IMAG=      0 | HEX=00000000

IFFT VERIFICATION
Maximum imaginary residual = 0
PASS: Output is effectively real.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>